# 03 — Feature Engineering

**Kenapa notebook ini ada?** Model scikit-learn hanya mengerti angka. Notebook ini
mengubah 15 kolom teks menjadi angka, membagi data latih/uji, dan menyiapkan
artefak yang dipakai notebook 05 serta dashboard.

**Tiga hal yang dikerjakan:**

1. Pisahkan fitur (`X`) dan target (`y`)
2. Bagi data latih/uji **sebelum** encoding dan scaling
3. Simpan hasilnya sebagai `.joblib`

## 0. Persiapan

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)

# notebook dijalankan dari folder notebooks/, jadi naik satu level ke root project
BASE = Path.cwd()
if not (BASE / "data").exists():
    BASE = BASE.parent
print("root project:", BASE)
import joblib

CLEAN = BASE / "data" / "clean" / "churn_clean.csv"
FEAT = BASE / "data" / "clean" / "churn_features.joblib"

df = pd.read_csv(CLEAN)
print("data bersih:", df.shape)

root project: c:\Users\53\Documents\data sains\machine learning\project ML Supervised
data bersih: (7043, 20)


## 1. Pisahkan fitur dan target

In [3]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

kolom_kategori = X.select_dtypes(include="str").columns.tolist()
kolom_numerik = X.select_dtypes(include=np.number).columns.tolist()

print("Kolom kategorikal (butuh encoding):", len(kolom_kategori))
print(kolom_kategori)
print()
print("Kolom numerik (dipakai langsung):", kolom_numerik)
print()
print("X:", X.shape, "| y:", y.shape)

Kolom kategorikal (butuh encoding): 15
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Kolom numerik (dipakai langsung): ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

X: (7043, 19) | y: (7043,)


## 2. Bagi data latih dan uji — **sebelum** encoding dan scaling

Ini urutan yang tidak boleh dibalik. Kalau kita meng-encode dan men-scale seluruh
data dulu baru membagi, informasi dari data uji ikut "bocor" ke proses training.
Akibatnya nilai evaluasi terlihat bagus tapi palsu. Istilahnya **data leakage**.

- `test_size=0.2` → 20% data disimpan untuk ujian akhir
- `stratify=y` → proporsi churn di train dan test dijaga sama (26,5%)
- `random_state=42` → hasil bisa direproduksi siapa pun

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("train:", X_train.shape, "| churn:", f"{y_train.mean():.2%}", f"({int(y_train.sum())} orang)")
print("test :", X_test.shape, "| churn:", f"{y_test.mean():.2%}", f"({int(y_test.sum())} orang)")

train: (5634, 19) | churn: 26.54% (1495 orang)
test : (1409, 19) | churn: 26.54% (374 orang)


## 3. Encoding — ubah teks jadi angka

Dipakai **One-Hot Encoding**: setiap kategori jadi kolom 0/1 sendiri.

Kenapa bukan Label Encoding (`Month-to-month` → 2, `Two year` → 0)? Karena Label
Encoding menciptakan urutan yang tidak ada artinya. Model akan menganggap
"Two year lebih besar dari One year", padahal itu cuma nama.

`drop="first"` membuang satu kolom per kategori untuk menghindari **dummy trap**
(kolom yang saling melengkapi sempurna, mengganggu Logistic Regression).

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

encoder = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), kolom_kategori),
    ],
    remainder="passthrough",   # kolom numerik dibiarkan apa adanya
)

X_train_enc = encoder.fit_transform(X_train)   # fit HANYA di train
X_test_enc = encoder.transform(X_test)         # test cuma transform

fitur = [f.replace("cat__", "").replace("remainder__", "")
         for f in encoder.get_feature_names_out()]

X_train_enc = pd.DataFrame(X_train_enc, columns=fitur, index=X_train.index)
X_test_enc = pd.DataFrame(X_test_enc, columns=fitur, index=X_test.index)

print("sebelum encoding:", X_train.shape[1], "kolom")
print("sesudah encoding:", X_train_enc.shape[1], "kolom")
print()
print("contoh kolom baru:")
for f in fitur[:8]:
    print("  -", f)

sebelum encoding: 19 kolom
sesudah encoding: 30 kolom

contoh kolom baru:
  - gender_Male
  - Partner_Yes
  - Dependents_Yes
  - PhoneService_Yes
  - MultipleLines_No phone service
  - MultipleLines_Yes
  - InternetService_Fiber optic
  - InternetService_No


19 kolom asli menjadi **30 fitur**. Penambahan ini normal: satu kolom teks dengan
3 kategori menjadi 2 kolom 0/1.

## 4. Scaling — hanya untuk model yang butuh

`StandardScaler` menyamakan skala fitur (rata-rata 0, simpangan baku 1). Ini
**wajib** untuk KNN (karena menghitung jarak) dan Logistic Regression (agar
koefisiennya adil), tapi **tidak perlu** untuk Random Forest (berbasis pemisahan
nilai, bukan jarak).

Sama seperti encoding: `fit` hanya di train, lalu dipakai untuk `transform` test.

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train_enc), columns=fitur, index=X_train.index)
X_test_sc = pd.DataFrame(scaler.transform(X_test_enc), columns=fitur, index=X_test.index)

print("sebelum scaling (3 fitur pertama):")
print(X_train_enc[fitur[:3]].head(3).round(2).to_string())
print()
print("sesudah scaling:")
print(X_train_sc[fitur[:3]].head(3).round(2).to_string())

sebelum scaling (3 fitur pertama):
      gender_Male  Partner_Yes  Dependents_Yes
3738          1.0          0.0             0.0
3151          1.0          1.0             1.0
4860          1.0          1.0             1.0

sesudah scaling:
      gender_Male  Partner_Yes  Dependents_Yes
3738         0.99        -0.97           -0.65
3151         0.99         1.03            1.53
4860         0.99         1.03            1.53


## 5. Simpan artefak

Satu file `.joblib` berisi semua yang dibutuhkan notebook 05 dan dashboard,
sehingga tidak ada yang perlu mengulang cleaning dari nol.

In [6]:
FEAT.parent.mkdir(parents=True, exist_ok=True)

joblib.dump({
    "X_train": X_train_enc, "X_test": X_test_enc,
    "X_train_scaled": X_train_sc, "X_test_scaled": X_test_sc,
    "y_train": y_train, "y_test": y_test,
    "features": fitur,
    "kolom_kategori": kolom_kategori,
    "encoder": encoder, "scaler": scaler,
}, FEAT)

print("disimpan:", FEAT)
print("ukuran  :", round(FEAT.stat().st_size / 1024, 1), "KB")

disimpan: C:\Users\53\Documents\data sains\machine learning\project ML Supervised\data\clean\churn_features.joblib
ukuran  : 3591.8 KB


## 6. Verifikasi — muat ulang dan bandingkan

Kalau hasil muat ulang berbeda dengan aslinya, berarti ada yang salah saat menyimpan.

In [7]:
cek = joblib.load(FEAT)
print("isi artefak:", list(cek.keys()))
print()
print("bentuk X_train      :", cek["X_train"].shape)
print("bentuk X_train_scaled:", cek["X_train_scaled"].shape)
print("bentuk X_test       :", cek["X_test"].shape)
print("jumlah baris y_train:", len(cek["y_train"]))
assert cek["X_train"].shape[0] == len(cek["y_train"])
assert list(cek["features"]) == fitur
print()
print("verifikasi lulus")

isi artefak: ['X_train', 'X_test', 'X_train_scaled', 'X_test_scaled', 'y_train', 'y_test', 'features', 'kolom_kategori', 'encoder', 'scaler']

bentuk X_train      : (5634, 30)
bentuk X_train_scaled: (5634, 30)
bentuk X_test       : (1409, 30)
jumlah baris y_train: 5634

verifikasi lulus


## 7. Ringkasan notebook 03

| Hal | Nilai |
|---|---|
| Data latih | 5.634 pelanggan (1.496 churn) |
| Data uji | 1.409 pelanggan (374 churn) |
| Kolom asli | 19 |
| Fitur setelah encoding | 30 |
| Encoding | One-Hot, `drop="first"`, fit di train |
| Scaling | StandardScaler, fit di train |
| Artefak | `data/clean/churn_features.joblib` |

Urutan yang dipegang sepanjang project:

```
split  →  encoding (fit train)  →  scaling (fit train)  →  model
```

Kalau urutan ini dibalik, semua angka evaluasi di notebook 05 tidak bisa dipercaya.